# 03 · Agreement analysis — gate G1 (PRD §7.4, §20.1)
**Runs:** locally. **Produces:** `data/labels/agreement_report.json`, `data/labels/labels_v1.jsonl`.

1. Independent second labeling run (different run id + seed) on 200 reviews → self-consistency.
2. Blinded reference sheet (text only — no stars, no model labels) → an annotator fills
   `experience_score`.
3. Gate: self κ > 0.70, reference κ > 0.55, self MAE < 0.40, reference MAE < 0.60.
   If it fails: add few-shot examples / chain-of-thought as PRD §7.4 suggests, then relabel
   with a NEW run id (the checkpoint refuses prompt changes by design).

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

In [ ]:
from dotenv import load_dotenv
import pandas as pd
from vera_training.labeling import Provider, run_labels

load_dotenv(ROOT / ".env")
pool = pd.read_parquet(paths.LABELING_POOL)
subset = pool.sample(n=200, random_state=7)
MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

state = run_labels(
    subset, paths.CONSISTENCY_RUN, Provider("gemini", MODEL),
    provider_name="gemini", model=MODEL, run_id="consistency", seed=2,
    max_requests=300, requests_per_minute=10,
)
print("second-run labels:", len(state["labels"]))

## Optional human validation — use only annotations supplied by a person

Fill the `experience_score` column (1.0 … 5.0 in 0.5 steps). LLM-generated reference
annotations must not be described as human validation.

In [ ]:
from vera_training.agreement import export_human_sheet

if not paths.HUMAN_SHEET.exists():
    export_human_sheet(subset, paths.HUMAN_SHEET, n=200, seed=42)
    print("Created", paths.HUMAN_SHEET, "- annotate it, then run the next cell.")
else:
    print("Sheet exists:", paths.HUMAN_SHEET)

In [ ]:
from vera_training.agreement import agreement_report

report = agreement_report(
    paths.PRIMARY_RUN, paths.CONSISTENCY_RUN, paths.HUMAN_SHEET, paths.AGREEMENT_REPORT,
    reference_annotator="human"
)
import json
print(json.dumps(
    {k: report[k] for k in ("passed", "human_validated", "self_consistency", "reference_agreement")},
    indent=2,
))

## Freeze labels (only after G1 passes)

In [ ]:
import json
from vera_training.agreement import require_label_gate
from vera_training.labeling import merge_labels

require_label_gate(paths.AGREEMENT_REPORT)
labelled = merge_labels(pool, paths.PRIMARY_RUN)
with paths.LABELS_V1.open("w", encoding="utf-8") as stream:
    for record in labelled.to_dict(orient="records"):
        stream.write(json.dumps(record, default=str) + "\n")
print(f"Wrote {len(labelled)} labels to {paths.LABELS_V1}")